# 04 · Gold aggregations (REQ-03)
**Purpose:** Summarise the Silver table into 3 small Delta tables, one per business question.
Gold is rebuilt from Silver on every run (overwrite), because the tables are small and always reflect all clean data.

In [0]:
%run ./00_config

In [0]:
from pyspark.sql import functions as F

# Open the clean Silver table (normal read, not streaming)
silver = spark.table(SILVER_TABLE)

print("Silver rows:", f"{silver.count():,}")

In [0]:
demand_counts = (
    silver
        .groupBy("day_type", "pickup_hour")
        .agg(
            F.count("*").alias("total_trips"),
            F.countDistinct("pickup_date").alias("num_days"),
        )
        .orderBy("day_type", "pickup_hour")
)

display(demand_counts)

In [0]:
demand_per_day = demand_counts.withColumn(
    "avg_trips_per_day",
    F.round(F.col("total_trips") / F.col("num_days"), 0)
)

display(demand_per_day)

In [0]:
# Save as a named Delta table (replace it every time we rebuild)
(
    demand_per_day
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_DEMAND_TABLE)
)

print("✅ Saved:", GOLD_DEMAND_TABLE)

In [0]:
display(spark.sql(f"SELECT * FROM {GOLD_DEMAND_TABLE} ORDER BY day_type, pickup_hour"))

In [0]:
total_rides = silver.count()   # all clean rides, used for the "share" column

payment_summary = (
    silver
        .groupBy("payment_method")
        .agg(
            F.count("*").alias("total_trips"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
            F.round(F.avg("tip_amount"), 2).alias("avg_tip"),
            F.round(F.avg("tip_percent"), 2).alias("avg_tip_percent"),
            F.round(F.avg(F.when(F.col("tip_amount") > 0, 1).otherwise(0)) * 100, 1)
                .alias("pct_trips_with_tip"),
        )
        .withColumn("share_of_trips_pct",
                    F.round(F.col("total_trips") / total_rides * 100, 2))
        .orderBy(F.desc("total_trips"))
)

display(payment_summary)

In [0]:
(
    payment_summary
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_PAYMENT_TABLE)
)

print("✅ Saved:", GOLD_PAYMENT_TABLE)
display(spark.sql(f"SELECT * FROM {GOLD_PAYMENT_TABLE}"))

In [0]:
silver = spark.table(SILVER_TABLE)

vendor_summary = (
    silver
        .groupBy("vendor_name")
        .agg(
            F.count("*").alias("total_trips")
        )
)

display(vendor_summary)

In [0]:
vendor_summary = (
    silver
        .groupBy("vendor_name")
        .agg(
            F.count("*").alias("total_trips"),
            F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
            F.round(F.avg("trip_distance"), 2).alias("avg_distance_miles"),
            F.round(F.avg("avg_speed_mph"), 2).alias("avg_speed_mph"),
            F.round(F.avg("tip_percent"), 2).alias("avg_tip_percent")
        )
)

display(vendor_summary)

In [0]:
total_rides = silver.count()

vendor_summary = vendor_summary.withColumn(
    "share_of_trips_pct",
    F.round(F.col("total_trips") / F.lit(total_rides) * 100, 2)
)

display(vendor_summary)


In [0]:
(
    vendor_summary
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(GOLD_VENDOR_TABLE)
)
print("✅ Saved:", GOLD_VENDOR_TABLE)
display(spark.sql(f"SELECT * FROM {GOLD_VENDOR_TABLE}"))

In [0]:
demand_pdf = spark.table(GOLD_DEMAND_TABLE).toPandas()
payment_pdf = spark.table(GOLD_PAYMENT_TABLE).toPandas()
vendor_pdf = spark.table(GOLD_VENDOR_TABLE).toPandas()

demand_pdf.to_csv("/Volumes/workspace/nyc_taxi/pipeline_files/gold_demand_by_hour.csv", index=False)
payment_pdf.to_csv("/Volumes/workspace/nyc_taxi/pipeline_files/gold_payment_tips.csv", index=False)
vendor_pdf.to_csv("/Volumes/workspace/nyc_taxi/pipeline_files/gold_vendor_comparison.csv", index=False)

print("Exported 3 Gold tables to CSV")

## Export small Gold tables for GitHub showcase

These Gold tables are tiny (a handful of rows each), so I export them as CSV to include in the repo - this lets a reviewer see real pipeline output without needing to run the whole pipeline themselves.